# VeriTune — LoRA Hyperparameter Sweep Analysis

Notebook 03: Analysing sweep results across LoRA rank, learning rate, and dropout.

**Key question:** Which (rank, lr, dropout) combination gives the best eval_loss  
per domain without excessive semantic drift or inference cost?

**Pareto trade-off:** accuracy vs cost vs latency

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))

from training.config import get_sweep_grid, get_domain_config, DOMAIN_DEFAULTS
from training.checkpoint_manager import CheckpointManager
from training.semantic_drift_tracker import SemanticDriftTracker, DriftResult

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
COLORS = {'technical': '#185FA5', 'billing': '#0F6E56', 'returns': '#993C1D', 'escalation': '#993556'}

## 1. Load sweep results

In [ ]:
DOMAINS = ['technical', 'billing', 'returns', 'escalation']
RESULTS_DIR = Path('../outputs/results')

def load_sweep_results(domain: str) -> pd.DataFrame:
    path = RESULTS_DIR / f'{domain}_sweep_results.json'
    if not path.exists():
        print(f'No sweep results found for {domain} — generating synthetic data for demo')
        return _synthetic_sweep_results(domain)
    with open(path) as f:
        records = json.load(f)
    rows = []
    for r in records:
        if 'error' in r:
            continue
        row = {
            'run_name':     r['run_name'],
            'lora_r':       r['lora_r'],
            'learning_rate': r['learning_rate'],
            'eval_loss':    r['metrics'].get('eval_loss', np.nan),
        }
        rows.append(row)
    return pd.DataFrame(rows)

def _synthetic_sweep_results(domain: str) -> pd.DataFrame:
    """Generate plausible synthetic sweep data for demonstration."""
    np.random.seed({'technical':0,'billing':1,'returns':2,'escalation':3}[domain])
    ranks = [8, 16, 32, 64]
    lrs   = [1e-4, 2e-4, 5e-4]
    rows  = []
    best_r = {'technical':32,'billing':24,'returns':28,'escalation':8}[domain]
    for r in ranks:
        for lr in lrs:
            # Simulate U-curve: sweet spot near best_r
            rank_penalty = abs(r - best_r) / max(best_r, 1) * 0.1
            lr_penalty   = abs(lr - 2e-4) / 2e-4 * 0.05
            base_loss    = 0.28 + rank_penalty + lr_penalty + np.random.randn() * 0.01
            rows.append({
                'lora_r': r, 'learning_rate': lr,
                'eval_loss': round(max(0.20, base_loss), 4),
                'run_name': f'{domain}_r{r}_lr{lr:.0e}'
            })
    return pd.DataFrame(rows)

dfs = {d: load_sweep_results(d) for d in DOMAINS}
for d, df in dfs.items():
    print(f'{d:12s}: {len(df)} runs | best eval_loss = {df["eval_loss"].min():.4f}')

## 2. Eval loss vs LoRA rank (all domains)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, domain in zip(axes, DOMAINS):
    df = dfs[domain]
    for lr, grp in df.groupby('learning_rate'):
        mean_loss = grp.groupby('lora_r')['eval_loss'].mean()
        ax.plot(mean_loss.index, mean_loss.values,
                marker='o', label=f'lr={lr:.0e}', linewidth=2)

    best_r = df.loc[df['eval_loss'].idxmin(), 'lora_r']
    best_loss = df['eval_loss'].min()
    ax.axvline(best_r, color=COLORS[domain], linestyle='--', alpha=0.5, label=f'Best r={best_r}')

    ax.set_title(f'{domain.capitalize()} (best r={best_r}, loss={best_loss:.4f})')
    ax.set_xlabel('LoRA rank (r)')
    ax.set_ylabel('eval_loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Eval Loss vs LoRA Rank by Domain & Learning Rate', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/results/sweep_rank_vs_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Pareto frontier: accuracy vs cost per ticket

In [ ]:
# Cost model: cost ∝ lora_r (more params = more compute per token)
# Accuracy proxy: 1 - normalised eval_loss

fig, ax = plt.subplots(figsize=(10, 6))

for domain in DOMAINS:
    df = dfs[domain].copy()
    # Normalise eval_loss to accuracy proxy [0,1]
    df['accuracy'] = 1 - (df['eval_loss'] - df['eval_loss'].min()) / (df['eval_loss'].max() - df['eval_loss'].min() + 1e-8)
    # Cost: rank-proportional (r=8 → $0.03, r=64 → $0.24)
    df['cost'] = df['lora_r'] / 8 * 0.03

    # Best per rank
    best_per_rank = df.groupby('lora_r').apply(lambda g: g.loc[g['eval_loss'].idxmin()])

    ax.scatter(best_per_rank['cost'], best_per_rank['accuracy'],
               c=COLORS[domain], s=80, zorder=5)
    ax.plot(best_per_rank['cost'].sort_values(),
            best_per_rank.set_index('cost')['accuracy'].sort_index(),
            color=COLORS[domain], linewidth=1.5, alpha=0.7, label=domain)

    # Label the optimal (best accuracy/cost trade-off)
    best_cfg = get_domain_config(domain)
    opt_cost = best_cfg.lora_r / 8 * 0.03
    ax.scatter([opt_cost], [0.85], marker='*', s=250, c=COLORS[domain], zorder=10)

ax.set_xlabel('Estimated cost per ticket ($)', fontsize=12)
ax.set_ylabel('Accuracy proxy (1 − norm. eval_loss)', fontsize=12)
ax.set_title('Pareto Frontier: Accuracy vs Cost per Ticket\n(★ = selected optimal config)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/results/pareto_frontier.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Learning rate sensitivity heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, domain in zip(axes, ['technical', 'escalation']):
    df = dfs[domain]
    pivot = df.pivot_table(index='lora_r', columns='learning_rate', values='eval_loss', aggfunc='mean')
    sns.heatmap(
        pivot, ax=ax, cmap='YlOrRd_r', annot=True, fmt='.3f',
        cbar_kws={'label': 'eval_loss'},
        linewidths=0.5,
    )
    ax.set_title(f'{domain.capitalize()} — eval_loss heatmap\n(rank × learning_rate)', fontsize=12)
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('LoRA Rank (r)')

plt.suptitle('Hyperparameter Sensitivity: Rank × Learning Rate', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/results/lr_sensitivity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Selected optimal configs summary

In [ ]:
rows = []
for domain in DOMAINS:
    df  = dfs[domain]
    cfg = get_domain_config(domain)
    best_run = df.loc[df['eval_loss'].idxmin()]
    rows.append({
        'Domain':           domain,
        'Selected r':       cfg.lora_r,
        'Selected α':       cfg.lora_alpha,
        'Selected LR':      f'{cfg.learning_rate:.0e}',
        'Best eval_loss':   f'{best_run["eval_loss"]:.4f}',
        'Est. cost/ticket': f'${cfg.lora_r / 8 * 0.03:.2f}',
        'Rationale':        {
            'technical':  'Higher rank for multi-step troubleshooting reasoning',
            'billing':    'Moderate rank; billing vocab is smaller',
            'returns':    'Policy-driven; slightly higher than billing',
            'escalation': 'Binary signal; tiny adapter avoids overfitting',
        }[domain]
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

## 6. Trainable parameter count per domain

In [ ]:
# Estimate LoRA parameter count: 2 × r × (d_model × target_modules)
# Mistral-7B: d_model=4096, typical LoRA targets 7 module pairs
D_MODEL = 4096
N_TARGET_PAIRS = 7   # q,k,v,o,gate,up,down

param_data = []
for domain in DOMAINS:
    cfg = get_domain_config(domain)
    n_lora_params = 2 * cfg.lora_r * D_MODEL * N_TARGET_PAIRS * 32  # ×32 layers
    base_params   = 7_241_748_480  # Mistral-7B total
    pct = n_lora_params / base_params * 100
    param_data.append({'domain': domain, 'lora_r': cfg.lora_r,
                       'trainable_params': n_lora_params,
                       'pct_of_base': pct})

param_df = pd.DataFrame(param_data)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(param_df['domain'], param_df['trainable_params'] / 1e6,
              color=[COLORS[d] for d in DOMAINS], alpha=0.85, edgecolor='white')

for bar, (_, row) in zip(bars, param_df.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'r={row["lora_r"]}\n{row["pct_of_base"]:.2f}%',
            ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Trainable parameters (M)')
ax.set_title('LoRA Trainable Parameter Count by Domain')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/results/trainable_params.png', dpi=150, bbox_inches='tight')
plt.show()

print(param_df[['domain','lora_r','trainable_params','pct_of_base']].to_string(index=False))